In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
import json, torch, os, numpy as np
from b2bTools.singleSeq.Predictor import MineSuite
from b2bTools.singleSeq import constants

_process_local = None

def _init_worker():
    global _process_local
    _process_local = MineSuite()

def process_one(j, save_dir):
    global _process_local
    minesuite = _process_local

    seqs = [("X", j["X"]), ("H", j["H"]), ("L", j["L"])]
    seqs_to_predict = [(name, seq) for name, seq in seqs if seq != ""]

    if seqs_to_predict:
        try:
            minesuite.predictSeqs(
                seqs_to_predict,
                predTypes=(
                    constants.TOOL_EFOLDMINE,
                    constants.TOOL_DISOMINE,
                    constants.TOOL_AGMATA,
                    constants.TOOL_PSP
                )
            )
        except Exception as e:
            print(f"Prediction failed for {j['pdb']}: {e}")
            pass  

    result = {}
    feature_names = ['backbone', 'sidechain', 'ppII', 'earlyFolding', 'disoMine']
    for name, seq in seqs:
        if not seq:
            result[name] = None
            continue
        try:
            pred = minesuite.allPredictions.get(name, {})
            agmata_arr = np.array(pred.get("agmata", []))
            if len(agmata_arr) == 0:
                result[name] = None
                continue
            seq_fea_i = agmata_arr[:, 1:].astype(np.float32)
            for f in feature_names:
                arr = np.array(pred.get(f, []))
                if len(arr) > 0:
                    feature_f = arr[:, 1:].astype(np.float32)
                    seq_fea_i = np.concatenate([seq_fea_i, feature_f], axis=1)
            result[name] = torch.tensor(seq_fea_i, dtype=torch.float32).T
        except Exception:
            result[name] = None

    torch.save(result, os.path.join(save_dir, j["pdb"] + ".pt"))
    return j["pdb"]

if __name__ == "__main__":
    save_dir = "/root/private_data/luog/Data_AbAg/gs_testabag_1215/ssf"
    os.makedirs(save_dir, exist_ok=True)

    with open("/root/private_data/luog/Data_AbAg/gs_testabag_1215/origin_data/training_data_final.json") as f:
        data = list(json.load(f))  

    max_workers = 2 

    ed_pt_files=[]
    for root, dirs, files in os.walk("/root/private_data/luog/Data_AbAg/gs_testabag_1215/ssf"):
        for file in files:
            if file.endswith(".pt"):
                ed_pt_files.append(file)
                
    print(f"Starting with {max_workers} workers...")
    with ProcessPoolExecutor(max_workers=max_workers, initializer=_init_worker) as executor:
        
        futures=[]
        for j in data:  
            if  j["pdb"]+".pt" not in ed_pt_files:
                futures.append(executor.submit(process_one, j, save_dir))
            else:
                print(f"already exists - {j['pdb']}")
                continue

        # tqdm
        success = 0
        for f in tqdm(as_completed(futures), total=len(futures), desc="Processing"):
            try:
                pdb_id = f.result()
                success += 1
            except Exception as e:
                print(f"Error: {e}")

    print(f"Done. Processed {success}/{len(data)} items.")